<a href="https://colab.research.google.com/github/rfandan/Transformers/blob/main/Vanilla_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
import wandb
from torch.amp import autocast, GradScaler

In [ ]:
num_samples = 500
seq_len = 50        # smaller than 200 → faster
d_model = 512

# ----------------------------
# DEVICE
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ----------------------------
# TOY DATA
# ----------------------------
# Random input
X = torch.randn(num_samples, seq_len, d_model)
# x: [batch_size(=30), seq_len, d_model]
y_target = torch.roll(X, shifts=-1, dims=1)
start = torch.zeros(num_samples, 1, d_model, device=X.device, dtype=X.dtype)
y_input = torch.cat([start, y_target[:, :-1, :]], dim=1)

# ----------------------------
# TRAIN / VAL / TEST SPLIT
# ----------------------------
train_split = int(0.7 * num_samples)
val_split   = int(0.85 * num_samples)

X_train, y_in_train, y_tgt_train = (X[:train_split],y_input[:train_split],y_target[:train_split])

X_val, y_in_val, y_tgt_val = (X[train_split:val_split],y_input[train_split:val_split],y_target[train_split:val_split])

X_test, y_in_test, y_tgt_test = (X[val_split:],y_input[val_split:],y_target[val_split:])

# ----------------------------
# DATASET CLASS
# ----------------------------
class CustomDataset(Dataset):
    def __init__(self, X, y_input, y_target):
        assert len(X) == len(y_input) == len(y_target)

        self.X = X.float()
        self.y_input = y_input.float()
        self.y_target = y_target.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_input[idx], self.y_target[idx]

# ----------------------------
# DATASETS
# ----------------------------
train_dataset = CustomDataset(X_train, y_in_train,y_tgt_train)
val_dataset   = CustomDataset(X_val, y_in_val,y_tgt_val)
test_dataset  = CustomDataset(X_test, y_in_test,y_tgt_test)

# ----------------------------
# DATALOADERS
# ----------------------------
generator = torch.Generator()
generator.manual_seed(42)

batch_size = 30
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
def scaled_dot_product(q, k, v, mask=None):
  # q, k, v = [30 x 8 x 200 x 64]
  d_k = q.size()[-1] # 64
  scaled = torch.matmul(q, k.transpose(-1,-2))/math.sqrt(d_k) #[30 x 8 x 200 x 64]*[30 x 8 x 64 x 200]=[30 x 8 x 200 x 200]
  if mask is not None:
    scaled = scaled.masked_fill(mask == 0, float("-inf")) # [30 x 8 x 200 x 200]
  attention = F.softmax(scaled,dim=-1) # [30 x 8 x 200 x 200]
  values = torch.matmul(attention, v) #[30 x 8 x 200 x 200]*[30 x 8 x 200 x 64]=[30 x 8 x 200 x 64]
  return values, attention # value->[30 x 8 x 200 x 64]; attention->[30 x 8 x 200 x 200]

class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model = d_model # 512
    self.num_heads = num_heads # 8
    self.head_dim = d_model // num_heads # 64
    self.qkv_layer = nn.Linear(d_model, 3*d_model) # [512 x 1536]
    self.linear_layer = nn.Linear(d_model, d_model) # [512 x 512]

  def forward(self,x,mask=None):
    batch_size, sequence_length, d_model = x.size() # [30 x 200 x 512]
    qkv = self.qkv_layer(x) # [30 x 200 x 1536]
    qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3*self.head_dim) # [30 x 200 x 8 x 192]
    qkv = qkv.permute(0, 2, 1, 3).contiguous() # [30 x 8 x 200 x 192]
    q, k, v = qkv.chunk(3, dim=-1) # q->[30 x 8 x 200 x 64], k->[30 x 8 x 200 x 64], v->[30 x 8 x 200 x 64]
    values, attention = scaled_dot_product(q, k, v, mask) # value->[30 x 8 x 200 x 64]; attention->[30 x 8 x 200 x 200]
    values = values.permute(0, 2, 1, 3).contiguous()
    values = values.reshape(batch_size, sequence_length, self.d_model) # [30 x 200 x 512] (8*64)
    out = self.linear_layer(values) # [30 x 200 x 512]
    return out #[30 x 200 x 512] same as input x but with more contextual awareness

class MultiHeadCrossAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model = d_model # 512
    self.num_heads = num_heads # 8
    self.head_dim = d_model // num_heads # 64
    self.kv_layer = nn.Linear(d_model, 2*d_model) # [512 x 1024]
    self.q_layer = nn.Linear(d_model, d_model) # [512 x 512]
    self.linear_layer = nn.Linear(d_model, d_model) # [512 x 512]

  def forward(self,x,y,mask=None): #x->input to encoder; y->input to decoder
    batch_size, src_len, _ = x.size() # [30 x 200]
    batch_size, tgt_len, _ = y.size() # [30 x 200]
    kv = self.kv_layer(x) # [30 x 200 x 1024] from encoder
    q = self.q_layer(y) # [30 x 200 x 512] from decoder
    kv = kv.reshape(batch_size, src_len, self.num_heads, 2*self.head_dim) # [30 x 200 x 8 x 128]
    kv = kv.permute(0, 2, 1, 3).contiguous() # [30 x 8 x 200 x 128]
    q = q.reshape(batch_size, tgt_len, self.num_heads, self.head_dim) # [30 x 200 x 8 x 64]
    q = q.permute(0, 2, 1, 3).contiguous() # [30 x 8 x 200 x 64]
    k, v = kv.chunk(2, dim=-1) # k->[30 x 8 x 200 x 64], v->[30 x 8 x 200 x 64]
    values, attention = scaled_dot_product(q, k, v, mask) # value->[30 x 8 x 200 x 64]; attention->[30 x 8 x 200 x 200]
    values = values.permute(0, 2, 1, 3).contiguous()
    values = values.reshape(batch_size, tgt_len, self.d_model) # [30 x 200 x 512] (8*64)
    out = self.linear_layer(values) # [30 x 200 x 512]
    return out #[30 x 200 x 512]

class LayerNorm(nn.Module):
  def __init__(self, parameters_shape, eps=1e-5):
    super().__init__()
    self.parameters_shape = parameters_shape #embedding dimension i.e., [512]
    self.eps = eps
    self.gamma = nn.Parameter(torch.ones(parameters_shape)) # gamma * x + beta [512] (learnable parameters)
    self.beta = nn.Parameter(torch.zeros(parameters_shape)) # [512]

  def forward(self, inputs): # inputs->[30 x 200 x 512]
    dims = [-(i+1) for i in range(len(self.parameters_shape))] # [-1]
    mean = inputs.mean(dim=dims, keepdim=True) # [30 x 200 x 1] mean of the last dimension
    var = ((inputs-mean)**2).mean(dim=dims, keepdim=True) # [30 x 200 x 1] variance
    std = (var+self.eps).sqrt() # [30 x 200 x 1] standard deviation
    y = (inputs-mean)/std # [30 x 200 x 512]
    out=self.gamma * y + self.beta # [30 x 200 x 512]
    return out

class FeedForwardLayer(nn.Module):
  def __init__(self,d_model,ffn_hidden,drop_prob):
    super().__init__()
    self.linear1 = nn.Linear(d_model,ffn_hidden) #[512,2048]
    self.linear2 = nn.Linear(ffn_hidden,d_model) #[2048,512]
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(p=drop_prob)

  def forward(self,x): # x->[30 x 200 x 512]
    x = self.linear1(x) # [30 x 200 x 2048]
    x = self.relu(x) # [30 x 200 x 2048]
    x = self.dropout(x) # [30 x 200 x 2048]
    x = self.linear2(x) # [30 x 200 x 512]
    return x


class EncoderLayer(nn.Module):
  def __init__(self,d_model,ffn_hidden,num_heads,drop_prob):
    super().__init__()
    self.attention = MultiHeadAttention(d_model,num_heads)
    self.norm1 = LayerNorm(parameters_shape=[d_model])
    self.dropout1 = nn.Dropout(p=drop_prob)
    self.ffn = FeedForwardLayer(d_model,ffn_hidden,drop_prob)
    self.norm2 = LayerNorm(parameters_shape=[d_model])
    self.dropout2 = nn.Dropout(p=drop_prob)

  def forward(self,x):
    residual = x # [30 x 200 x 512]
    x = self.attention(x,mask=None) # [30 x 200 x 512]
    x = self.dropout1(x) # [30 x 200 x 512]
    x = self.norm1(x+residual) # [30 x 200 x 512]
    residual = x # [30 x 200 x 512]
    x = self.ffn(x) # [30 x 200 x 512]
    x = self.dropout2(x) # [30 x 200 x 512]
    x = self.norm2(x+residual)
    return x # output of one encoder, same as input x but with more contextual awareness

class Encoder(nn.Module):
  def __init__(self,d_model,ffn_hidden,num_heads,drop_prob,num_layers):
    super().__init__()
    self.layers = nn.Sequential(*[EncoderLayer(d_model,ffn_hidden,num_heads,drop_prob) for _ in range(num_layers)])

  def forward(self,x):
    x = self.layers(x)
    return x # output after n encoder, same as input x but with more contextual awareness

def causal_mask(seq_len, device):
    return torch.tril(torch.ones(seq_len, seq_len, device=device))

class DecoderLayer(nn.Module):
  def __init__(self,d_model,ffn_hidden,num_heads,drop_prob):
    super().__init__()
    self.self_attention = MultiHeadAttention(d_model, num_heads)
    self.norm1 = LayerNorm(parameters_shape=[d_model])
    self.dropout1 = nn.Dropout(p=drop_prob)
    self.encoder_decoder_attention = MultiHeadCrossAttention(d_model,num_heads)
    self.norm2 = LayerNorm(parameters_shape=[d_model])
    self.dropout2 = nn.Dropout(p=drop_prob)
    self.ffn = FeedForwardLayer(d_model,ffn_hidden,drop_prob)
    self.norm3 = LayerNorm(parameters_shape=[d_model])
    self.dropout3 = nn.Dropout(p=drop_prob)

  def forward(self,x,y,decoder_mask=None):
    _y = y
    seq_len = y.size(1)
    if decoder_mask is None:
      decoder_mask = causal_mask(seq_len, y.device)
    decoder_mask = decoder_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, T, T]
    y = self.self_attention(y,mask=decoder_mask) # [30 x 200 x 512]
    y = self.dropout1(y)
    y = self.norm1(y+_y) # [30 x 200 x 512]
    _y = y # [30 x 200 x 512]
    y = self.encoder_decoder_attention(x,y,mask=None) # [30 x 200 x 512]
    y = self.dropout2(y)
    y = self.norm2(y+_y) # [30 x 200 x 512]
    _y = y # [30 x 200 x 512]
    y = self.ffn(y) # [30 x 200 x 512]
    y = self.dropout3(y)
    y = self.norm3(y+_y) # [30 x 200 x 512]
    return y


class SequentialDecoder(nn.Sequential):
  def forward(self, *inputs):
    x, y, mask = inputs
    for module in self._modules.values():
      y = module(x, y, mask) # [30 x 200 x 512]
    return y

class Decoder(nn.Module):
  def __init__(self,d_model,ffn_hidden,num_heads,drop_prob,num_layers):
    super().__init__()
    self.layers = SequentialDecoder(*[DecoderLayer(d_model,ffn_hidden,num_heads,drop_prob) for _ in range(num_layers)])

  def forward(self,x,y,mask=None): #x->input sequence; y-> output sequence e.g. x:English, y:Latvian
    # x->[30 x 200 x 512]
    # mask->[200 x 200]
    y = self.layers(x,y,mask) # y->[30 x 200 x 512]
    return y

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

class Transformer(nn.Module):
    def __init__(self,d_model,ffn_hidden,num_heads,drop_prob,num_encoder_layers,num_decoder_layers,max_len=500):
        super().__init__()
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        # Encoder & Decoder
        self.encoder = Encoder(d_model, ffn_hidden, num_heads, drop_prob, num_encoder_layers)
        self.decoder = Decoder(d_model, ffn_hidden, num_heads, drop_prob, num_decoder_layers)

    def forward(self, x, y, mask=None):
        # x: source  [batch, seq_len, d_model]
        # y: target  [batch, seq_len, d_model]
        # Add positional encoding
        x = self.pos_encoding(x)
        y = self.pos_encoding(y)
        # Encoder
        enc_out = self.encoder(x)
        # Decoder
        out = self.decoder(enc_out, y, mask)
        return out


In [ ]:
# ============================================================
# TRAINER CLASS (GENERAL MLP VERSION: CLASSIFICATION + REGRESSION)
# ============================================================

class Trainer:
    def __init__(
        self,
        model,
        optimizer,
        criterion,
        device,

        train_loader,
        val_loader=None,
        test_loader=None,

        scheduler=None,
        metrics=None,

        epochs=10,
        patience=5,
        grad_clip=1.0,

        use_amp=True,
        compile_model=False,

        use_wandb=False,
        project_name="trainer-exp",

        log_every_n_epochs=1
    ):
        # ============================================================
        # DEVICE + MODEL
        # ============================================================
        self.device = device
        self.model = model.to(device)

        if compile_model:
            self.model = torch.compile(self.model)

        # ============================================================
        # CORE COMPONENTS
        # ============================================================
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler

        # ============================================================
        # DATA
        # ============================================================
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

        # ============================================================
        # TRAIN CONFIG
        # ============================================================
        self.epochs = epochs
        self.patience = patience
        self.grad_clip = grad_clip
        self.log_every_n_epochs = log_every_n_epochs

        # ============================================================
        # AMP (MIXED PRECISION)
        # ============================================================
        self.use_amp = use_amp and device.type == "cuda"
        self.scaler = GradScaler(enabled=self.use_amp)

        # ============================================================
        # METRICS
        # ============================================================
        # NOTE:
        # metrics must be compatible with task (regression OR classification)
        self.metrics = {k: v.to(device) for k, v in metrics.items()} if metrics else {}
        self.train_metrics = {k: copy.deepcopy(v).to(device) for k, v in metrics.items()} if metrics else {}
        self.val_metrics = {k: copy.deepcopy(v).to(device) for k, v in metrics.items()} if metrics else {}

        # ============================================================
        # EARLY STOPPING
        # ============================================================
        self.best_val_loss = float("inf")
        self.early_stop_counter = 0

        # ============================================================
        # W&B
        # ============================================================
        self.use_wandb = use_wandb
        if self.use_wandb:
            wandb.init(
                project=project_name,
                config={
                    "epochs": epochs,
                    "patience": patience,
                    "grad_clip": grad_clip,
                    "use_amp": self.use_amp,
                    "compile_model": compile_model
                }
            )

    # ============================================================
    # TRAIN ONE EPOCH
    # ============================================================
    def _train_one_epoch(self):
        self.model.train()

        total_loss = 0.0

        for metric in self.train_metrics.values():
            metric.reset()

        for x, y_in, y_tgt in self.train_loader:
            x = x.to(self.device, non_blocking=True)
            y_in = y_in.to(self.device, non_blocking=True)
            y_tgt = y_tgt.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            # ----------------------------
            # FORWARD PASS
            # ----------------------------
            with autocast(device_type=self.device.type, enabled=self.use_amp):
                outputs = self.model(x,y_in)
                loss = self.criterion(outputs, y_tgt)

            # ----------------------------
            # BACKWARD PASS
            # ----------------------------
            self.scaler.scale(loss).backward()

            # ----------------------------
            # GRADIENT CLIPPING
            # ----------------------------
            if self.grad_clip is not None:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.grad_clip
                )

            # ----------------------------
            # OPTIMIZER STEP
            # ----------------------------
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # ----------------------------
            # METRICS UPDATE
            # ----------------------------
            for metric in self.train_metrics.values():
                metric.update(outputs.detach().reshape(-1), y_tgt.detach().reshape(-1))

            total_loss += loss.item()

            # ----------------------------
            # SCHEDULER STEP
            # ----------------------------
            if self.scheduler:
                self.scheduler.step()

        train_loss = total_loss / len(self.train_loader)

        train_results = {
            k: v.compute().item()
            for k, v in self.train_metrics.items()
        }

        return train_loss, train_results

    # ============================================================
    # VALIDATION
    # ============================================================
    def _validate(self):
        if self.val_loader is None:
            return None, None

        self.model.eval()

        total_loss = 0.0

        for metric in self.val_metrics.values():
            metric.reset()

        with torch.inference_mode():
            for x, y_in, y_tgt in self.val_loader:
                x = x.to(self.device)
                y_in = y_in.to(self.device)
                y_tgt = y_tgt.to(self.device)

                with autocast(device_type=self.device.type, enabled=self.use_amp):
                    outputs = self.model(x,y_in)
                    loss = self.criterion(outputs, y_tgt)

                for metric in self.val_metrics.values():
                    metric.update(outputs.detach().reshape(-1), y_tgt.detach().reshape(-1))

                total_loss += loss.item()

        val_loss = total_loss / len(self.val_loader)

        val_results = {
            k: v.compute().item()
            for k, v in self.val_metrics.items()
        }

        return val_loss, val_results

    # ============================================================
    # FIT LOOP
    # ============================================================
    def fit(self):
        self.best_val_loss = float("inf")
        self.early_stop_counter = 0

        for epoch in range(self.epochs):

            train_loss, train_metrics = self._train_one_epoch()
            val_loss, val_metrics = self._validate()

            current_lr = self.optimizer.param_groups[0]["lr"]

            # ============================================================
            # EARLY STOPPING
            # ============================================================
            if val_loss is not None:
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.early_stop_counter = 0

                    torch.save({
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": self.optimizer.state_dict(),
                        "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
                        "best_val_loss": self.best_val_loss,
                    }, "best_model.pt")

                else:
                    self.early_stop_counter += 1

                if self.early_stop_counter >= self.patience:
                    print("🛑 Early stopping triggered")
                    break

            # Save last model
            torch.save({
                "epoch": epoch,
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
            }, "last_model.pt")

            # ============================================================
            # W&B LOGGING
            # ============================================================
            if self.use_wandb:
                log_dict = {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss if val_loss is not None else 0,
                    "lr": current_lr,
                }

                for k, v in train_metrics.items():
                    log_dict[f"train/{k}"] = v

                for k, v in val_metrics.items():
                    log_dict[f"val/{k}"] = v

                wandb.log(log_dict)

            # ============================================================
            # CONSOLE LOGGING
            # ============================================================
            if (epoch + 1) % self.log_every_n_epochs == 0:
                val_loss_str = f"{val_loss:.4f}" if val_loss is not None else "N/A"
                print(
                    f"Epoch {epoch+1}/{self.epochs} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Val Loss: {val_loss_str} | "
                    f"LR: {current_lr:.6f}"
                )

        print("✅ Training finished")

    # ============================================================
    # TEST
    # ============================================================
    def test(self):
        if self.test_loader is None:
            return

        checkpoint = torch.load("best_model.pt", map_location=self.device, weights_only=True)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        self.model.eval()

        total_loss = 0.0

        for metric in self.metrics.values():
            metric.reset()

        with torch.inference_mode():
            for x, y_in, y_tgt in self.test_loader:
                x = x.to(self.device)
                y_in = y_in.to(self.device)
                y_tgt = y_tgt.to(self.device)

                outputs = self.model(x,y_in)
                loss = self.criterion(outputs, y_tgt)

                total_loss += loss.item()

                for metric in self.metrics.values():
                    metric.update(outputs.detach().reshape(-1), y_tgt.detach().reshape(-1))

        results = {k: m.compute().item() for k, m in self.metrics.items()}

        print(f"\nTest Loss: {total_loss / len(self.test_loader):.4f}")
        for k, v in results.items():
            print(f"{k}: {v:.4f}")

In [ ]:
!pip install torchmetrics -q
from torchmetrics import MeanSquaredError, MeanAbsoluteError, R2Score

metrics = {"mse": MeanSquaredError()}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.7 MB/s eta 0:00:00


In [ ]:
model = Transformer(
    d_model=512,
    ffn_hidden=2048,
    num_heads=8,
    drop_prob=0.1,
    num_encoder_layers=1,
    num_decoder_layers=1
)

In [ ]:
print(model)

Transformer(
  (pos_encoding): PositionalEncoding()
  (encoder): Encoder(
    (layers): Sequential(
      (0): EncoderLayer(
        (attention): MultiHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNorm()
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): FeedForwardLayer(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (relu): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNorm()
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Decoder(
    (layers): SequentialDecoder(
      (0): DecoderLayer(
        (self_attention): MultiHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_la

In [ ]:
criterion = nn.SmoothL1Loss()  # Huber loss
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0.01)

EPOCHS = 10

# Create Scheduler OUTSIDE the Trainer
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy="cos",
    div_factor=25,
    final_div_factor=1e4
)

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,

    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,

    log_every_n_epochs=1,

    scheduler=scheduler,
    metrics=metrics,
    epochs=EPOCHS, #IMPORTANT: epochs in both trainer and schedular should be same
    patience=10,
    use_amp=True,
    use_wandb=False
)

In [ ]:
trainer.fit()

Epoch 1/10 | Train Loss: 0.5465 | Val Loss: 0.5220 | LR: 0.000293
Epoch 2/10 | Train Loss: 0.5076 | Val Loss: 0.4802 | LR: 0.000784
Epoch 3/10 | Train Loss: 0.4786 | Val Loss: 0.4687 | LR: 0.001000
Epoch 4/10 | Train Loss: 0.4719 | Val Loss: 0.4672 | LR: 0.000942
Epoch 5/10 | Train Loss: 0.4690 | Val Loss: 0.4660 | LR: 0.000797
Epoch 6/10 | Train Loss: 0.4677 | Val Loss: 0.4652 | LR: 0.000593
Epoch 7/10 | Train Loss: 0.4669 | Val Loss: 0.4647 | LR: 0.000371
Epoch 8/10 | Train Loss: 0.4660 | Val Loss: 0.4643 | LR: 0.000174
Epoch 9/10 | Train Loss: 0.4656 | Val Loss: 0.4641 | LR: 0.000042
Epoch 10/10 | Train Loss: 0.4653 | Val Loss: 0.4641 | LR: 0.000000
✅ Training finished


In [ ]:
trainer.test()


Test Loss: 0.4642
mse: 1.8559


In [ ]:
text = "The cat walked over the road"

text_encoding = {s:i for i,s in enumerate(sorted(sentence.replace(',', '').split()))}
print(text_encoding)

In [ ]:
text_to_tensor = torch.tensor([text_encoding[s] for s in text.split()])
print(text_to_tensor)

In [ ]:
vocab_size = 50000
d_model = 6
torch.manual_seed(42)
embedding_layer = nn.Embedding(vocab_size, 6)
print(embedding_layer(text_to_tensor))
print(embedding_layer(text_to_tensor).shape)